In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from rag.PdfProcessor import PdfProcessor
from rag.vector_store.VectorStoreIngestor import VectorStoreIngestor 
from huggingface_hub import HfApi, login

load_dotenv()
file_path = os.getenv("FILE_PATH")
acess_token = os.getenv("ACESS_TOKEN")
api = HfApi()
login(acess_token)

if not file_path:
    raise ValueError("FILE_PATH não está definido no arquivo .env")

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Arquivo não encontrado: {file_path}")

print(f"Carregando PDF de: {file_path}")

In [2]:
ingestor = VectorStoreIngestor(
    persist_dir="rag/vector-store/db/chroma"  
)

vectorstore = ingestor.load_vectorstore()
doc_count = vectorstore._collection.count()

if doc_count == 0:
    print("⚠️ Banco vazio! Ingerindo PDF...")
    vectorstore = ingestor.ingest_pdf(file_path)
    doc_count = vectorstore._collection.count()
    print(f"✅ {doc_count} documentos ingeridos!")
else:
    print(f"✅ ChromaDB carregado com {doc_count} documentos")


c:\Users\marcos.stanquini\OneDrive - PERPETUO CONSULTORIA ESPECIALIZADA LTDA\Área de Trabalho\Work\RAG\rag\vector_store\VectorStoreIngestor.py:33: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_function = SentenceTransformerEmbeddings(
c:\Users\marcos.stanquini\OneDrive - PERPETUO CONSULTORIA ESPECIALIZADA LTDA\Área de Trabalho\Work\RAG\rag\vector_store\VectorStoreIngestor.py:59: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchai

⚠️ Banco vazio! Ingerindo PDF...
📄 Processando PDF: rag/data/PPCENGSOFTWARE.pdf
✅ Total de chunks gerados: 385
💾 Armazenando chunks no ChromaDB...
✅ Total de chunks gerados: 385
💾 Armazenando chunks no ChromaDB...
✅ Ingestão concluída! Dados persistidos em: rag/vector-store/db/chroma
✅ 385 documentos ingeridos!
✅ Ingestão concluída! Dados persistidos em: rag/vector-store/db/chroma
✅ 385 documentos ingeridos!


In [3]:

user_query = "O que fala a materia Algoritmos e programação 1"


top_3_chunks = vectorstore.similarity_search(user_query, k=3)

print(f"📄 Encontrados {len(top_3_chunks)} chunks relevantes:\n")
for i, chunk in enumerate(top_3_chunks, 1):
    print(f"{i}. Página {chunk.metadata.get('page')} - {len(chunk.page_content)} caracteres")


📄 Encontrados 3 chunks relevantes:

1. Página 63 - 761 caracteres
2. Página 22 - 849 caracteres
3. Página 47 - 831 caracteres


In [ ]:

from huggingface_hub import InferenceClient

context = "\n\n---\n\n".join([
    f"[Página {chunk.metadata.get('page')}]\n{chunk.page_content}" 
    for chunk in top_3_chunks
])

print("📝 Contexto montado!\n")

prompt = f"""Com base no contexto abaixo, responda a pergunta de forma clara e objetiva.

CONTEXTO:
{context}

PERGUNTA: {user_query}

RESPOSTA:"""

messages = [{"role": "user", "content": prompt}]
client = InferenceClient("meta-llama/Meta-Llama-3-8B-Instruct")
response = client.chat_completion(messages, max_tokens=600, temperature=0.3)

print("\n=== RESPOSTA DO MODELO ===")
print(response.choices[0].message.content)


📝 Contexto montado!


=== RESPOSTA DO MODELO ===
A matéria "Algoritmos e Programação 1" aborda os conceitos básicos sobre computadores, com foco no desenvolvimento de algoritmos. Ela visa desenvolver o raciocínio lógico voltado à programação de computadores, desenvolver a lógica de programação e as técnicas de programação estruturada, e especificar, implementar, compilar, executar e testar programas utilizando uma linguagem de programação de alto nível.

=== RESPOSTA DO MODELO ===
A matéria "Algoritmos e Programação 1" aborda os conceitos básicos sobre computadores, com foco no desenvolvimento de algoritmos. Ela visa desenvolver o raciocínio lógico voltado à programação de computadores, desenvolver a lógica de programação e as técnicas de programação estruturada, e especificar, implementar, compilar, executar e testar programas utilizando uma linguagem de programação de alto nível.
